#  Transformer Model Fine-Tuning for Duplicate Question Detection

## Objective

Build and fine-tune a **DistilBERT** transformer model using Hugging Face Transformers to classify duplicate question pairs from the Quora Duplicate Questions Dataset. This week focuses on:

- Implementing transformer-based architecture for semantic similarity
- Fine-tuning DistilBERT on sentence pair classification
- Comparing performance with Week 2 baseline (TF-IDF + Logistic Regression)
- Evaluating the model using standard NLP metrics
- Creating production-ready inference pipeline
- Saving trained model for future deployment

**Dataset:** Quora Duplicate Questions Dataset (sentence-transformers/quora-duplicates)
**Model:** distilbert-base-uncased
**Framework:** Hugging Face Transformers + PyTorch + Trainer API

## 1. Import Libraries


In [ ]:
# Core scientific computing libraries
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Install Trainer dependency before importing Transformers
import sys
import subprocess
import importlib.metadata

try:
    importlib.metadata.version('accelerate')
except importlib.metadata.PackageNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'accelerate>=1.1.0'])

# PyTorch and Transformers
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Hugging Face Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import load_dataset, Dataset

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress tokenizer parallelism warnings
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print("All libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"Device available: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## 2. GPU Configuration

In [ ]:
# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    torch.cuda.empty_cache()
else:
    print("GPU not available. Training will use CPU (significantly slower).")

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("GPU configuration complete")

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
# Create output directories
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
TRANSFORMER_RESULTS_DIR = PROJECT_ROOT / 'results' / 'transformer_model'
MODELS_DIR = PROJECT_ROOT / 'models'

TRANSFORMER_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directories ready: {TRANSFORMER_RESULTS_DIR} and {MODELS_DIR}")

## 3. Load Preprocessed Dataset

In [ ]:
# Load cleaned Quora dataset from parquet
print("Loading cleaned Quora Duplicate Questions dataset from parquet...")
df = pd.read_parquet(DATA_DIR / 'cleaned_quora.parquet')

# Split into 80% training and 20% test
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# Rename columns to match expected format (questions -> [q1, q2], is_duplicate -> label)
train_df = train_df.rename(columns={'clean_q1': 'q1', 'clean_q2': 'q2', 'label': 'is_duplicate'})
test_df = test_df.rename(columns={'clean_q1': 'q1', 'clean_q2': 'q2', 'label': 'is_duplicate'})

# Keep only necessary columns
train_df = train_df[['q1', 'q2', 'is_duplicate']]
test_df = test_df[['q1', 'q2', 'is_duplicate']]

print(f"\nDataset Information:")
print(f"Training set size: {len(train_df):,} pairs")
print(f"Test set size: {len(test_df):,} pairs")
print(f"\nColumn names: {train_df.columns.tolist()}")
print(f"\nFirst 3 samples:")
print(train_df.head(3))

# Check class distribution
print(f"\nClass Distribution (Training Set):")
print(train_df['is_duplicate'].value_counts())
print(f"\nClass Distribution (Test Set):")
print(test_df['is_duplicate'].value_counts())

# Check for missing values
print(f"\nMissing values in training set:")
print(train_df.isnull().sum())
print(f"\nMissing values in test set:")
print(test_df.isnull().sum())

## 4. Transformer Architecture Selection

In [ ]:
# Model Configuration
MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 128  # Maximum sequence length for tokenization
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
WARMUP_STEPS = 500
 
 
print(f"Model Configuration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Max Sequence Length: {MAX_LENGTH}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Number of Epochs: {NUM_EPOCHS}")
print(f"  Warmup Steps: {WARMUP_STEPS}")
print(f"  Device: {device}")

# Why DistilBERT?
print(f"\nWhy DistilBERT?")
print("  - 40% smaller than BERT, 60% faster inference")
print("  - Retains 97% of BERT's language understanding capabilities")
print("  - Efficient for production deployment")
print("  - Ideal for computational constraint environments")
print("  - Pre-trained on English corpus with bidirectional context")

## 5. Tokenization for Sentence Pair Classification

In [ ]:
# Load tokenizer
print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"\nTokenizer Information:")
print(f"  Vocabulary size: {tokenizer.vocab_size:,}")
print(f"  Special tokens: {tokenizer.special_tokens_map}")

# Example tokenization for sentence pair
example_question1 = "What is machine learning?"
example_question2 = "What is artificial intelligence?"

print(f"\nExample Tokenization (Sentence Pair):")
print(f"Question 1: {example_question1}")
print(f"Question 2: {example_question2}")

example_tokens = tokenizer(
    example_question1,
    example_question2,
    max_length=MAX_LENGTH,
    padding='max_length',
    truncation=True,
    return_tensors='pt'
)

print(f"\nTokenized output shape: {example_tokens['input_ids'].shape}")
print(f"Input IDs (first 20): {example_tokens['input_ids'][0][:20].tolist()}")
print(f"Attention Mask (first 20): {example_tokens['attention_mask'][0][:20].tolist()}")

## 6. Dataset Encoding Pipeline

In [ ]:
# Convert pandas DataFrames to Hugging Face Dataset format
from datasets import Dataset

print("Converting to Hugging Face Dataset format...")
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Tokenization function for dataset (updated for parquet columns)
def tokenize_function(examples):
    """
    Tokenize sentence pairs for DistilBERT.
    Handles maximum sequence length and padding.
    """
    return tokenizer(
        examples['q1'],  # First question (cleaned)
        examples['q2'],  # Second question (cleaned)
        max_length=MAX_LENGTH,
        padding='max_length',
        truncation=True,
    )

# Prepare datasets
print("Encoding training dataset...")
train_dataset_encoded = train_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=1000,
    remove_columns=['q1', 'q2'],
    desc="Tokenizing training data"
)

print("Encoding test dataset...")
test_dataset_encoded = test_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=1000,
    remove_columns=['q1', 'q2'],
    desc="Tokenizing test data"
)

# Rename 'is_duplicate' to 'labels' for Trainer API
train_dataset_encoded = train_dataset_encoded.rename_column('is_duplicate', 'labels')
test_dataset_encoded = test_dataset_encoded.rename_column('is_duplicate', 'labels')

# Set data format for PyTorch
train_dataset_encoded.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset_encoded.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f"\nEncoded Dataset Information:")
print(f"Training set: {len(train_dataset_encoded)} samples")
print(f"Test set: {len(test_dataset_encoded)} samples")
print(f"\nSample encoded data:")
print(train_dataset_encoded[0])

## 7. Train Validation Test Split

In [ ]:
# Split training data into train and validation (90-10 split)
print(f"Creating train-validation split...")
validation_fraction = 0.1
train_val_split = train_dataset_encoded.train_test_split(test_size=validation_fraction, seed=42)

train_dataset_final = train_val_split['train']
validation_dataset = train_val_split['test']

print(f"\nData Split Summary:")
print(f"Training set: {len(train_dataset_final):,} samples")
print(f"Validation set: {len(validation_dataset):,} samples")
print(f"Test set: {len(test_dataset_encoded):,} samples")
print(f"Total: {len(train_dataset_final) + len(validation_dataset) + len(test_dataset_encoded):,} samples")

# Class distribution in each split
train_labels = np.array(train_dataset_final['labels'])
val_labels = np.array(validation_dataset['labels'])
test_labels = np.array(test_dataset_encoded['labels'])

print(f"\nClass Distribution:")
print(f"Training - Duplicates: {(train_labels == 1).sum():,}, Non-duplicates: {(train_labels == 0).sum():,}")
print(f"Validation - Duplicates: {(val_labels == 1).sum():,}, Non-duplicates: {(val_labels == 0).sum():,}")
print(f"Test - Duplicates: {(test_labels == 1).sum():,}, Non-duplicates: {(test_labels == 0).sum():,}")
# Visualize class distribution across splits
fig, ax = plt.subplots(figsize=(8, 4))
split_names = ['Train', 'Validation', 'Test']
dup_counts = [(train_labels == 1).sum(), (val_labels == 1).sum(), (test_labels == 1).sum()]
non_dup_counts = [(train_labels == 0).sum(), (val_labels == 0).sum(), (test_labels == 0).sum()]
x = np.arange(len(split_names))
ax.bar(x - 0.2, non_dup_counts, 0.4, label='Non-Duplicate', color='steelblue')
ax.bar(x + 0.2, dup_counts, 0.4, label='Duplicate', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(split_names)
ax.set_title('Class Distribution Across Splits')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig(TRANSFORMER_RESULTS_DIR / 'split_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {TRANSFORMER_RESULTS_DIR / 'split_class_distribution.png'}")


## 8. Model Initialization

In [ ]:
# Load pre-trained model
print(f"Loading pre-trained model: {MODEL_NAME}...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2  # Binary classification: 0=non-duplicate, 1=duplicate
)

# Move model to device
model = model.to(device)

print(f"\nModel Architecture:")
print(model)

# Model parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
non_trainable_params = total_params - trainable_params

print(f"\nModel Size:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Non-trainable parameters: {non_trainable_params:,}")

## 9. Evaluation Metrics

In [ ]:
# Install Hugging Face evaluate if it is missing
import sys
import subprocess

try:
    import evaluate
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'evaluate'])
    import evaluate

print('evaluate is ready')

In [ ]:
# Import evaluation metrics from Hugging Face
import evaluate

# Load metrics
accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    """
    Compute evaluation metrics for model validation and testing.
    
    Metrics:
    - Accuracy: Overall correctness
    - F1 Score: Harmonic mean of precision and recall
    - Precision: Correctness of positive predictions
    - Recall: Coverage of positive instances
    - ROC-AUC: Area under the ROC curve
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)['accuracy']
    f1 = f1_metric.compute(predictions=predictions, references=labels, average='weighted')['f1']
    
    try:
        from scipy.special import softmax as scipy_softmax
        probs = scipy_softmax(logits, axis=1)[:, 1]
        auc = roc_auc_score(labels, probs)
    except Exception:
        auc = 0.0
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision_score(labels, predictions, average='weighted', zero_division=0),
        'recall': recall_score(labels, predictions, average='weighted', zero_division=0),
        'roc_auc': auc,
    }

print('Evaluation metrics configured')
print(f'\\nMetrics to be computed:')
print(f'  - Accuracy: Proportion of correct predictions')
print(f'  - Precision: True positives / (True positives + False positives)')
print(f'  - Recall: True positives / (True positives + False negatives)')
print(f'  - F1 Score: Harmonic mean of precision and recall')

### Why Use Evaluation Metrics?

Evaluation metrics help assess model performance beyond just accuracy. They provide different perspectives on how well the model makes predictions.

**Key Reasons:**
1. **Accuracy alone is misleading** - With imbalanced datasets, high accuracy doesn't mean good predictions
2. **Business implications** - Different mistakes have different costs
3. **Model comparison** - Metrics help compare different models objectively
4. **Identify bias** - Metrics show if model performs better on one class than another

**Our 4 Metrics Explained:**

| Metric | Formula | What it Measures | When to Use |
|--------|---------|------------------|------------|
| **Accuracy** | (TP+TN)/(TP+TN+FP+FN) | Overall correctness | Balanced datasets |
| **Precision** | TP/(TP+FP) | Of predicted duplicates, how many are actually duplicates? | When False Positives are costly |
| **Recall** | TP/(TP+FN) | Of actual duplicates, how many did we find? | When False Negatives are costly |
| **F1 Score** | 2*(Precision*Recall)/(Precision+Recall) | Harmonic mean balancing precision and recall | Best overall metric for imbalanced data |

**For Duplicate Question Detection:**
- **High Recall** = Find most duplicate questions (don't miss duplicates)
- **High Precision** = Avoid false alarms (don't flag non-duplicates as duplicates)
- **F1 Score** = Best balance between the two

## 10. Training Configuration

In [ ]:
# Create training arguments
import inspect
import sys
import importlib.metadata

print(f"Python executable: {sys.executable}")
print(f"Transformers version: {importlib.metadata.version('transformers')}")
print(f"Accelerate version: {importlib.metadata.version('accelerate')}")
print(f"Torch version: {torch.__version__}")

output_dir = MODELS_DIR / 'transformer_training_outputs'
Path(output_dir).mkdir(parents=True, exist_ok=True)

base_training_args = {
    'output_dir': output_dir,
    'num_train_epochs': NUM_EPOCHS,
    'per_device_train_batch_size': BATCH_SIZE,
    'per_device_eval_batch_size': BATCH_SIZE,
    'warmup_steps': WARMUP_STEPS,
    'weight_decay': 0.01,
    'learning_rate': LEARNING_RATE,
    'logging_steps': 100,
    'save_strategy': 'epoch',
    'load_best_model_at_end': True,
    'metric_for_best_model': 'f1',
    'greater_is_better': True,
    'save_total_limit': 3,
    'fp16': torch.cuda.is_available(),
    'gradient_accumulation_steps': 2,
    'seed': 42,
    'report_to': [],
}

training_arg_names = set(inspect.signature(TrainingArguments.__init__).parameters)
if 'evaluation_strategy' in training_arg_names:
    base_training_args['evaluation_strategy'] = 'epoch'
elif 'eval_strategy' in training_arg_names:
    base_training_args['eval_strategy'] = 'epoch'

supported_training_args = {
    key: value
    for key, value in base_training_args.items()
    if key in training_arg_names
}

ignored_args = sorted(set(base_training_args) - set(supported_training_args))
if ignored_args:
    print(f'Ignoring unsupported TrainingArguments for this environment: {ignored_args}')

try:
    training_args = TrainingArguments(**supported_training_args)
except ImportError as exc:
    if "accelerate" in str(exc):
        raise RuntimeError(
            "Accelerate is installed, but this notebook kernel loaded Transformers before it was available. "
            "Restart the kernel, then run the notebook from the top."
        ) from exc
    raise

print("Training Arguments:")
print(f"  Output directory: {output_dir}")
print(f"  Number of epochs: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Warmup steps: {WARMUP_STEPS}")
print("  Weight decay: 0.01")
print("  Evaluation strategy: epoch")
print(f"  FP16 (mixed precision): {torch.cuda.is_available()}")
print("  Gradient accumulation steps: 2")

## 11. Transformer Fine-Tuning

In [ ]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_final,
    eval_dataset=validation_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.0)],
)

print("Starting transformer fine-tuning...")
print(f"Training on {device}")
print(f"Total training steps: {len(train_dataset_final) // BATCH_SIZE * NUM_EPOCHS}")
print("\n" + "="*80)

# Train the model
train_result = trainer.train()

print("\n" + "="*80)
print("Training completed!")
print(f"\nTraining Results:")
print(f"  Final training loss: {train_result.training_loss:.4f}")
print(f"  Total training time: {train_result.metrics.get('train_runtime', 0):.2f} seconds")

## 12. Model Evaluation

In [ ]:
# Evaluate on validation set
print("Evaluating on validation set...")
val_results = trainer.evaluate()
print(f"\nValidation Metrics:")
for key, value in val_results.items():
    if not key.startswith('eval_runtime'):
        print(f"  {key}: {value:.4f}")

# Evaluate on test set
print("\nEvaluating on test set...")
test_results = trainer.evaluate(eval_dataset=test_dataset_encoded)
print(f"\nTest Set Metrics:")
for key, value in test_results.items():
    if not key.startswith('eval_runtime'):
        print(f"  {key}: {value:.4f}")

# Get predictions on test set
print("\nGenerating predictions on test set...")
predictions = trainer.predict(test_dataset_encoded)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = test_dataset_encoded['labels']

# Detailed classification report
print("\nDetailed Classification Report (Test Set):")
print(classification_report(
    true_labels,
    pred_labels,
    target_names=['Non-Duplicate', 'Duplicate'],
    digits=4
))

# Confusion matrix
cm = confusion_matrix(true_labels, pred_labels)
print(f"\nConfusion Matrix:")
print(f"  [[TN={cm[0,0]:6d}  FP={cm[0,1]:6d}]")
print(f"   [FN={cm[1,0]:6d}  TP={cm[1,1]:6d}]]")

## 13. Training Visualization

In [ ]:
# Extract training history
training_logs = trainer.state.log_history

# Parse logs into epochs
epochs = []
train_losses = []
val_losses = []
val_accuracies = []
val_f1_scores = []

for log in training_logs:
    if 'epoch' in log:
        if 'loss' in log and 'eval_loss' not in log:
            train_losses.append(log['loss'])
        if 'eval_loss' in log:
            epochs.append(log['epoch'])
            val_losses.append(log['eval_loss'])
            val_accuracies.append(log.get('eval_accuracy', 0))
            val_f1_scores.append(log.get('eval_f1', 0))

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss plot
if train_losses and val_losses:
    axes[0, 0].plot(range(1, len(train_losses) + 1), train_losses, 'o-', label='Training Loss', linewidth=2)
    axes[0, 0].plot(epochs, val_losses, 's-', label='Validation Loss', linewidth=2)
    axes[0, 0].set_xlabel('Epoch', fontsize=11)
    axes[0, 0].set_ylabel('Loss', fontsize=11)
    axes[0, 0].set_title('Training and Validation Loss', fontsize=12, fontweight='bold')
    axes[0, 0].legend(fontsize=10)
    axes[0, 0].grid(True, alpha=0.3)

# Accuracy plot
if val_accuracies:
    axes[0, 1].plot(epochs, val_accuracies, 'o-', color='green', linewidth=2, markersize=8)
    axes[0, 1].set_xlabel('Epoch', fontsize=11)
    axes[0, 1].set_ylabel('Accuracy', fontsize=11)
    axes[0, 1].set_title('Validation Accuracy', fontsize=12, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].set_ylim([0, 1])

# F1 Score plot
if val_f1_scores:
    axes[1, 0].plot(epochs, val_f1_scores, 's-', color='purple', linewidth=2, markersize=8)
    axes[1, 0].set_xlabel('Epoch', fontsize=11)
    axes[1, 0].set_ylabel('F1 Score', fontsize=11)
    axes[1, 0].set_title('Validation F1 Score', fontsize=12, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].set_ylim([0, 1])

# Confusion matrix heatmap
cm = confusion_matrix(true_labels, pred_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 1],
            xticklabels=['Non-Dup', 'Duplicate'],
            yticklabels=['Non-Dup', 'Duplicate'])
axes[1, 1].set_ylabel('True Label', fontsize=11)
axes[1, 1].set_xlabel('Predicted Label', fontsize=11)
axes[1, 1].set_title('Test Set Confusion Matrix', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(TRANSFORMER_RESULTS_DIR / 'transformer_training_visualization.png', dpi=300, bbox_inches='tight')
print(f"Training visualization saved to {TRANSFORMER_RESULTS_DIR / 'transformer_training_visualization.png'}")
plt.show()

## 14. Prediction Examples and Error Analysis

In [ ]:
# Function for making predictions on new text
def predict_duplicate(question1, question2):
    """
    Predict if two questions are duplicates.
    
    Args:
        question1: First question string
        question2: Second question string
    
    Returns:
        Dictionary with prediction and confidence
    """
    # Tokenize
    inputs = tokenizer(
        question1,
        question2,
        max_length=MAX_LENGTH,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    # Move to device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Get model prediction
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
    
    # Apply softmax for probabilities
    probs = torch.softmax(logits, dim=1)[0]
    predicted_class = torch.argmax(probs).item()
    confidence = probs[predicted_class].item()
    
    return {
        'question1': question1,
        'question2': question2,
        'prediction': 'Duplicate' if predicted_class == 1 else 'Non-Duplicate',
        'confidence': confidence,
        'is_duplicate': predicted_class
    }

# Test on sample predictions
print("Sample Predictions:")
print("="*100)

test_cases = [
    ("What is machine learning?", "What is artificial intelligence?"),
    ("How do I learn Python?", "How to learn Python programming?"),
    ("What is the capital of France?", "What is the largest city in France?"),
    ("How to make pasta?", "How to cook pizza?"),
    ("What is deep learning?", "What is neural networks?"),
]

predictions_list = []
for q1, q2 in test_cases:
    result = predict_duplicate(q1, q2)
    predictions_list.append(result)
    print(f"\nQ1: {q1}")
    print(f"Q2: {q2}")
    print(f"Prediction: {result['prediction']} (Confidence: {result['confidence']:.4f})")
    print("-"*100)

# ── Real Error Analysis ──
print("\n" + "="*100)
print("ERROR ANALYSIS")
print("="*100)

# Build a DataFrame with original questions for display
error_analysis_df = pd.DataFrame({
    'true_label': true_labels,
    'predicted_label': pred_labels,
    'q1': test_df['q1'].values,
    'q2': test_df['q2'].values,
    'confidence_dup': predictions.predictions[:, 1],
})

false_positives = error_analysis_df[
    (error_analysis_df['true_label'] == 0) & (error_analysis_df['predicted_label'] == 1)
].head(5)

false_negatives = error_analysis_df[
    (error_analysis_df['true_label'] == 1) & (error_analysis_df['predicted_label'] == 0)
].head(5)

print(f"\nTotal errors: {len(error_analysis_df[error_analysis_df['true_label'] != error_analysis_df['predicted_label']])}")

print("\n=== FALSE POSITIVES (predicted Duplicate, actually Non-Duplicate) ===")
for _, row in false_positives.iterrows():
    print(f"Q1: {row['q1']}\nQ2: {row['q2']}\nConfidence (dup): {row['confidence_dup']:.4f}\n{'-'*80}")

print("\n=== FALSE NEGATIVES (predicted Non-Duplicate, actually Duplicate) ===")
for _, row in false_negatives.iterrows():
    print(f"Q1: {row['q1']}\nQ2: {row['q2']}\nConfidence (dup): {row['confidence_dup']:.4f}\n{'-'*80}")


## 15. Comparison with Week 2 Baseline

In [ ]:
# Week 2 Baseline Results (from baseline_model.ipynb)
# These are the best results achieved with TF-IDF + Logistic Regression
baseline_results = {
    'Method': 'TF-IDF + Logistic Regression',
    'Accuracy': 0.7416,
    'Precision': 0.7128,
    'Recall': 0.5030,
    'F1 Score': 0.5911,
}

# Week 3 Transformer Results
transformer_results = {
    'Method': 'DistilBERT + Trainer',
    'Accuracy': test_results['eval_accuracy'],
    'Precision': test_results['eval_precision'],
    'Recall': test_results['eval_recall'],
    'F1 Score': test_results['eval_f1'],
}

# Create comparison dataframe
comparison_df = pd.DataFrame([
    baseline_results,
    transformer_results
])

print("\n" + "="*100)
print("WEEK 2 vs WEEK 3 PERFORMANCE COMPARISON")
print("="*100)
print(comparison_df.to_string(index=False))

# Calculate improvements
print("\n" + "="*100)
print("IMPROVEMENTS (Week 3 vs Week 2)")
print("="*100)

improvements = {
    'Accuracy': (transformer_results['Accuracy'] - baseline_results['Accuracy']) * 100,
    'Precision': (transformer_results['Precision'] - baseline_results['Precision']) * 100,
    'Recall': (transformer_results['Recall'] - baseline_results['Recall']) * 100,
    'F1 Score': (transformer_results['F1 Score'] - baseline_results['F1 Score']) * 100,
}

for metric, improvement in improvements.items():
    direction = "Improvement" if improvement > 0 else "Decline"
    print(f"{metric:15} {improvement:+.2f}% ({direction})")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Metrics comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
baseline_vals = [baseline_results[m] for m in metrics]
transformer_vals = [transformer_results[m] for m in metrics]

x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, baseline_vals, width, label='Week 2 (TF-IDF + LR)', color='skyblue', edgecolor='black')
axes[0].bar(x + width/2, transformer_vals, width, label='Week 3 (DistilBERT)', color='lightcoral', edgecolor='black')
axes[0].set_xlabel('Metrics', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Score', fontsize=12, fontweight='bold')
axes[0].set_title('Week 2 vs Week 3 Performance Comparison', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend(fontsize=10)
axes[0].set_ylim([0, 1])
axes[0].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, (b, t) in enumerate(zip(baseline_vals, transformer_vals)):
    axes[0].text(i - width/2, b + 0.02, f'{b:.3f}', ha='center', fontsize=9)
    axes[0].text(i + width/2, t + 0.02, f'{t:.3f}', ha='center', fontsize=9)

# Improvement percentage
improvement_vals = [improvements[m] for m in metrics]
colors = ['green' if v > 0 else 'red' for v in improvement_vals]
axes[1].bar(metrics, improvement_vals, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Improvement (%)', fontsize=12, fontweight='bold')
axes[1].set_title('Performance Improvement (Week 3 vs Week 2)', fontsize=13, fontweight='bold')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axes[1].grid(True, alpha=0.3, axis='y')

# Add value labels
for i, (metric, val) in enumerate(zip(metrics, improvement_vals)):
    axes[1].text(i, val + (0.5 if val > 0 else -1.5), f'{val:+.2f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(TRANSFORMER_RESULTS_DIR / 'week2_vs_week3_comparison.png', dpi=300, bbox_inches='tight')
print(f"\nComparison visualization saved to {TRANSFORMER_RESULTS_DIR / 'week2_vs_week3_comparison.png'}")
plt.show()

# Save comparison to CSV
comparison_df.to_csv(TRANSFORMER_RESULTS_DIR / 'week2_vs_week3_comparison.csv', index=False)
print(f"Comparison results saved to {TRANSFORMER_RESULTS_DIR / 'week2_vs_week3_comparison.csv'}")

## 16. Inference Function

In [ ]:
class DuplicateQuestionDetector:
    """
    Production-ready inference pipeline for duplicate question detection.
    
    This class wraps the fine-tuned DistilBERT model and provides:
    - Single prediction interface
    - Batch prediction for multiple question pairs
    - Confidence scores
    - Probability distributions
    """
    
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.model.eval()  # Set to evaluation mode
    
    def predict_single(self, question1, question2):
        """
        Predict if two questions are duplicates.
        
        Args:
            question1: First question string
            question2: Second question string
        
        Returns:
            dict with prediction, confidence, and probabilities
        """
        # Tokenize input
        inputs = self.tokenizer(
            question1,
            question2,
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Move to device
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        # Forward pass
        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)[0]
        
        predicted_class = torch.argmax(probs).item()
        confidence = probs[predicted_class].item()
        
        return {
            'is_duplicate': bool(predicted_class),
            'confidence': float(confidence),
            'prob_non_duplicate': float(probs[0].item()),
            'prob_duplicate': float(probs[1].item()),
        }
    
    def predict_batch(self, question_pairs):
        """
        Predict for multiple question pairs.
        
        Args:
            question_pairs: List of tuples [(q1, q2), ...]
        
        Returns:
            List of prediction dictionaries
        """
        results = []
        for q1, q2 in question_pairs:
            result = self.predict_single(q1, q2)
            result['question1'] = q1
            result['question2'] = q2
            results.append(result)
        return results

# Initialize detector
detector = DuplicateQuestionDetector(model, tokenizer, device)

# Test batch prediction
print("Testing Batch Prediction:")
print("="*100)

test_pairs = [
    ("What is Python?", "What is a programming language?"),
    ("How to learn machine learning?", "How to learn ML?"),
    ("What is the weather today?", "Is it raining?"),
]

batch_results = detector.predict_batch(test_pairs)

for i, result in enumerate(batch_results, 1):
    print(f"\nPair {i}:")
    print(f"Q1: {result['question1']}")
    print(f"Q2: {result['question2']}")
    print(f"Duplicate: {result['is_duplicate']}")
    print(f"Confidence: {result['confidence']:.4f}")
    print(f"Probability (Non-Duplicate): {result['prob_non_duplicate']:.4f}")
    print(f"Probability (Duplicate): {result['prob_duplicate']:.4f}")

print("\nInference pipeline ready for production")

## 17. Save Final Model

In [ ]:
# Create model output directory
model_output_dir = MODELS_DIR / 'saved_transformer_model'
Path(model_output_dir).mkdir(parents=True, exist_ok=True)

# Save model
print(f"Saving model to {model_output_dir}...")
model.save_pretrained(model_output_dir)
tokenizer.save_pretrained(model_output_dir)

print("Model saved successfully")
print(f"\nSaved files:")
for file in Path(model_output_dir).glob('*'):
    print(f"  - {file.name}")

# Save predictions on test set
print(f"\nSaving test set predictions...")
predictions_df = pd.DataFrame({
    'true_label': true_labels,
    'predicted_label': pred_labels,
    'confidence_non_dup': predictions.predictions[:, 0],
    'confidence_dup': predictions.predictions[:, 1],
})

predictions_df['max_confidence'] = predictions_df[['confidence_non_dup', 'confidence_dup']].max(axis=1)
predictions_csv_path = TRANSFORMER_RESULTS_DIR / 'transformer_test_predictions.csv'
predictions_df.to_csv(predictions_csv_path, index=False)
print(f"Predictions saved to {predictions_csv_path}")
print(f"  Sample predictions:")
print(predictions_df.head(10))

# Save confusion matrix
print(f"\nSaving confusion matrix...")
cm_df = pd.DataFrame(
    cm,
    index=['Predicted Non-Dup', 'Predicted Dup'],
    columns=['Actual Non-Dup', 'Actual Dup']
)
cm_csv_path = TRANSFORMER_RESULTS_DIR / 'transformer_confusion_matrix.csv'
cm_df.to_csv(cm_csv_path)
print(f"Confusion matrix saved to {cm_csv_path}")
print(cm_df)

## 18. Conclusion

In [ ]:
print("="*100)
print("WEEK 3 PROJECT SUMMARY: TRANSFORMER-BASED DUPLICATE QUESTION DETECTION")
print("="*100)

summary_text = f"""
PROJECT COMPLETION REPORT
{'='*100}

OVERVIEW:
  This project implemented a DistilBERT transformer model for duplicate question
  detection on the Quora Duplicate Questions Dataset. The model was fine-tuned using
  Hugging Face Transformers with a PyTorch backend.

KEY ACHIEVEMENTS:

1. DATA PREPARATION:
   - Dataset: {len(train_dataset_final):,} training samples (90% split)
   - Validation Set: {len(validation_dataset):,} samples
   - Test Set: {len(test_dataset_encoded):,} samples
   - Tokenization: DistilBERT tokenizer with max_length={MAX_LENGTH}

2. MODEL ARCHITECTURE:
   - Model: distilbert-base-uncased
   - Task: Binary sequence classification (Duplicate vs Non-Duplicate)
   - Parameters: {trainable_params:,} trainable

3. TRAINING CONFIGURATION:
   - Epochs: {NUM_EPOCHS}
   - Batch Size: {BATCH_SIZE}
   - Learning Rate: {LEARNING_RATE}
   - Optimizer: AdamW with weight decay
   - Device: {device}

4. FINAL PERFORMANCE (Test Set):
   - Accuracy:  {test_results['eval_accuracy']:.4f} ({test_results['eval_accuracy']*100:.2f}%)
   - Precision: {test_results['eval_precision']:.4f}
   - Recall:    {test_results['eval_recall']:.4f}
   - F1 Score:  {test_results['eval_f1']:.4f}

5. COMPARISON WITH WEEK 2 BASELINE:
   - Baseline (TF-IDF + Logistic Regression): Accuracy={baseline_results['Accuracy']:.4f}, F1={baseline_results['F1 Score']:.4f}
   - Week 3 (DistilBERT): Accuracy={test_results['eval_accuracy']:.4f}, F1={test_results['eval_f1']:.4f}
   - Improvement: Accuracy +{improvements['Accuracy']:.2f}%, F1 +{improvements['F1 Score']:.2f}%

6. OUTPUTS GENERATED:
   - Fine-tuned model saved in models/saved_transformer_model/
   - Training outputs saved in models/transformer_training_outputs/
   - Training visualization saved in results/transformer_model/transformer_training_visualization.png
   - Comparison plot saved in results/transformer_model/week2_vs_week3_comparison.png
   - Test predictions saved in results/transformer_model/transformer_test_predictions.csv
   - Confusion matrix saved in results/transformer_model/transformer_confusion_matrix.csv
   - Project summary saved in results/transformer_model/week3_project_summary.txt

7. PRODUCTION READINESS:
   - Model and tokenizer saved for deployment
   - Inference pipeline is available through DuplicateQuestionDetector
   - Batch prediction support is included
   - Confidence scores and probability distributions are available

{'='*100}
Week 3 Project Completed Successfully!
{'='*100}
"""

print(summary_text)

# Save summary to file
with open(TRANSFORMER_RESULTS_DIR / 'week3_project_summary.txt', 'w') as f:
    f.write(summary_text)
print(f"\nSummary saved to {TRANSFORMER_RESULTS_DIR / 'week3_project_summary.txt'}")

## 19. Updated requirements.txt dependencies

In [ ]:
# Updated requirements.txt for Week 3 with transformer dependencies
requirements_content = """pandas
numpy
matplotlib
seaborn
scipy
nltk
gensim
scikit-learn
pyarrow
sentence-transformers
torch>=2.0.0
transformers>=4.30.0
datasets>=2.12.0
evaluate>=0.4.0
python-docx
accelerate>=1.1.0
"""

# Write to requirements.txt
requirements_path = 'requirements.txt'
with open(requirements_path, 'w') as f:
    f.write(requirements_content.strip())

print(f"Updated {requirements_path}:")
print("="*60)
print(requirements_content)
print("="*60)
print(f"\n{requirements_path} updated with Week 3 dependencies")

print(f"\nNew dependencies added for Week 3:")
new_deps = [
    "transformers>=4.30.0  # Hugging Face Transformers library",
    "datasets>=2.12.0      # Hugging Face Datasets library",
    "evaluate>=0.4.0       # Evaluation metrics library",
    "accelerate>=1.1.0     # Distributed training support",
    "torch>=2.0.0          # PyTorch",
]
for dep in new_deps:
    print(f"  - {dep}")

print(f"\nTo install updated dependencies, run:")
print(f"  pip install -r requirements.txt")

In [ ]:
import os

print("=== RESULTS FOLDER ===")
print(os.listdir(TRANSFORMER_RESULTS_DIR))

print("\n=== MODELS FOLDER ===")
print(os.listdir(MODELS_DIR / 'saved_transformer_model'))

print("\n=== TRAINING COMPLETE ===")
print("All outputs saved successfully!")